# CrowdWisdomTrading: Bitcoin Sentiment & Multi-Horizon Price Forecasting
### Quantitative Data Scientist Internship Assessment
**100% Free-First & Reproducible Implementation**

This notebook demonstrates:
1. Ingestion of real BTC-USD hourly prices and VADER-scored social sentiment.
2. Point-in-time feature engineering with zero lookahead bias.
3. Multi-horizon target formulation ($n+0$ to $n+5$).
4. Chronological walk-forward cross-validation across 4 models.
5. Out-of-sample trading strategy simulation and risk metrics.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add project root to sys.path
ROOT_DIR = Path.cwd().parent
sys.path.insert(0, str(ROOT_DIR))

from config.settings import get_config
from src.database import DatabaseManager

cfg = get_config()
db = DatabaseManager(cfg.database_path)
print(f"Connected to SQLite Database: {cfg.database_path}")

## 1. Load Market Prices & Aggregated Sentiment from SQLite

In [ ]:
prices_df = db.get_prices(ticker="BTC-USD", frequency="1h")
agg_sent = db.get_aggregated_sentiment(frequency="1h")
posts_df = db.get_x_posts(ticker="BTC")

print(f"Prices shape: {prices_df.shape}")
print(f"Aggregated sentiment shape: {agg_sent.shape}")
print(f"Total X posts: {len(posts_df)}")
prices_df.tail()

## 2. Inspect Out-of-Sample Predictions across 6 Horizons

In [ ]:
preds_df = db.get_predictions()
print(f"Total prediction records: {len(preds_df)}")
preds_df.head(10)

## 3. Compare Out-of-Sample Evaluation Results

In [ ]:
eval_results = db.get_evaluation_results()
summary = eval_results[["horizon", "model_name", "mae", "rmse", "directional_accuracy", "baseline_mae", "sharpe_ratio"]]
summary.drop_duplicates(subset=["horizon", "model_name"]).sort_values(["horizon", "mae"])